# DESARROLLO DE MODELOS DE MACHINE LEARNING

Desarrollado Por: 
* Daniel Arturo Martinez Morales 
* Luis Miguel Ortega Cañate

A continuacion se presenta el desarrollo de la fase de entrenamiento  y test de los modelos  de machine learning  para el desarrollo de un Sistema Clasificador de Resultados de Análisis de Aceite Usado en Motores de Maquinaria Pesada.

**Nota:** Para el desarrollo de esta fase se tiene en cuenta ls fase de exploración de los datos desarrolado previamente en el Noteboock llamado "Exploracion de datos".

## Balanceo de Muestras 

Durante la fase de exploración de datos se identificó un desbalance significativo
en la variable objetivo `Assigned Condition Rating`, donde la clase Normal concentra
la gran mayoría de las observaciones, mientras que las clases Advertencia y Crítico
representan una proporción considerablemente menor del total de muestras.

Este desbalance es inherente al dominio del problema: en operación normal, la mayor
parte de los componentes de una flota minera se encuentran en condición saludable,
y los eventos de advertencia o falla crítica son relativamente poco frecuentes.

Sin embargo, este desbalance introduce
los siguientes riesgos si no es tratado:

- El modelo tiende a clasificar la mayoría de muestras como Normal para maximizar
  el accuracy global, ignorando las clases minoritarias.
- Se genera un sesgo hacia la clase mayoritaria, reduciendo drásticamente el
  desempeño  para las clases Advertencia y Crítico, que son precisamente las
  de mayor valor operativo para el negocio.
- Las métricas globales como el Accuracy resultan engañosas, ocultando el
  bajo desempeño real del modelo sobre las condiciones anómalas.

Detectar una condición Crítica que el modelo clasifica como Normal representa un Falso Negativo de alto costo operativo, asociado directamente a las pérdidas de ~500 kUSD/año documentadas en el planteamiento del problema.


In [5]:
import pandas as pd
data_cleaned = pd.read_excel("../Exploración/Analisis De Aceite Flotas Mineras.xlsx")

In [6]:
print("=== Distribución original de clases ===")
print(data_cleaned['Assigned Condition Rating'].value_counts())
print(data_cleaned['Assigned Condition Rating'].value_counts(normalize=True)
      .mul(100).round(2).astype(str) + ' %')

=== Distribución original de clases ===
Assigned Condition Rating
Normal      3607
Warning      522
Critical     154
Name: count, dtype: int64
Assigned Condition Rating
Normal      84.22 %
Warning     12.19 %
Critical      3.6 %
Name: proportion, dtype: object


In [12]:
import pandas as pd
import numpy as np


# CALIFICACIÓN SACODE - Generación de Límites Condenatorios (Metodología Noria)


## Definición de Predictores según cada Categoría
Salud = data_cleaned[['Boro (ppm)', 'Calcio (ppm)', 'Cinc (ppm)', 'Fosforado (ppm)',
                       'Magnesio (ppm)', 'Molibdeno (ppm)', 'Nitración JOAP (Abs/cm)',
                       'Oxidación JOAP (Abs/cm)', 'Sulfatación JOAP (Abs/cm)', 'Sodio (ppm)']]

Desgaste = data_cleaned[['Aluminio (ppm)', 'Cobre (ppm)', 'Cromo (ppm)', 'Estaño (ppm)',
                          'Hierro (ppm)', 'Plomo (ppm)',
                          'Oxidación JOAP (Abs/cm)']]

Contaminacion = data_cleaned[['Agua (%)', 'Dilución por combustible (%)',
                               'Hollín JOAP (Abs/cm)', 'Silicio (ppm)',
                               'Sodio (ppm)', 'Aluminio (ppm)']]



def calcular_limites(df: pd.DataFrame, 
                     factor_warning: float = 2.0, 
                     factor_critico: float = 3.0) -> pd.DataFrame:

    stats = pd.DataFrame({
        'media'           : df.mean(),
        'sigma'           : df.std(),
        'lim_warn_sup'    : df.mean() + factor_warning * df.std(),
        'lim_warn_inf'    : df.mean() - factor_warning * df.std(),
        'lim_critico_sup' : df.mean() + factor_critico * df.std(),
        'lim_critico_inf' : df.mean() - factor_critico * df.std(),
    })

    stats['lim_warn_inf']    = stats['lim_warn_inf'].clip(lower=0)
    stats['lim_critico_inf'] = stats['lim_critico_inf'].clip(lower=0)

    return stats



def clasificar_predictor(valor: float, 
                         lim_warn_sup: float, 
                         lim_critico_sup: float) -> str:

    if valor <= lim_warn_sup:
        return 'Normal'
    elif valor <= lim_critico_sup:
        return 'Advertencia'
    else:
        return 'Crítico'



# Función: Calificación SACODE por categoría (criterio de peor caso)


def calificar_categoria(df: pd.DataFrame, limites: pd.DataFrame) -> pd.Series:

    orden_severidad = {'Normal': 0, 'Advertencia': 1, 'Crítico': 2}
    inv_orden       = {v: k for k, v in orden_severidad.items()}

    calificaciones = pd.DataFrame(index=df.index)

    for col in df.columns:
        lim_w = limites.loc[col, 'lim_warn_sup']
        lim_c = limites.loc[col, 'lim_critico_sup']
        calificaciones[col] = df[col].apply(
            lambda x: clasificar_predictor(x, lim_w, lim_c)
        )

    # Peor caso: máximo nivel de severidad entre todos los predictores por fila
    peor_caso = calificaciones.map(lambda x: orden_severidad[x]).max(axis=1)

    return peor_caso.map(inv_orden)



# Cálculo de Límites por Categoría


limites_salud        = calcular_limites(Salud)
limites_desgaste     = calcular_limites(Desgaste)
limites_contaminacion = calcular_limites(Contaminacion)

print("=== Límites Salud ===")
print(limites_salud.round(3), "\n")

print("=== Límites Desgaste ===")
print(limites_desgaste.round(3), "\n")

print("=== Límites Contaminación ===")
print(limites_contaminacion.round(3), "\n")

# Asignación de Calificación SACODE al dataset


data_cleaned['SACODE_Salud']         = calificar_categoria(Salud,         limites_salud)
data_cleaned['SACODE_Desgaste']      = calificar_categoria(Desgaste,      limites_desgaste)
data_cleaned['SACODE_Contaminacion'] = calificar_categoria(Contaminacion, limites_contaminacion)

=== Límites Salud ===
                              media    sigma  lim_warn_sup  lim_warn_inf  \
Boro (ppm)                   57.419   39.573       136.565         0.000   
Calcio (ppm)               2045.069  690.602      3426.273       663.865   
Cinc (ppm)                 1351.095  149.697      1650.489      1051.702   
Fosforado (ppm)            1179.651   78.502      1336.655      1022.647   
Magnesio (ppm)              241.588  142.551       526.691         0.000   
Molibdeno (ppm)              53.587   14.047        81.680        25.494   
Nitración JOAP (Abs/cm)       5.791    1.364         8.518         3.063   
Oxidación JOAP (Abs/cm)      13.620    2.617        18.855         8.385   
Sulfatación JOAP (Abs/cm)    18.457    2.613        23.683        13.231   
Sodio (ppm)                   5.874   47.363       100.600         0.000   

                           lim_critico_sup  lim_critico_inf  
Boro (ppm)                         176.139            0.000  
Calcio (ppm)     

In [13]:

# Limite Warning sodio
limites_salud.loc['Sodio (ppm)', 'lim_warn_sup'] = 40
# Limite Critico sodio
limites_salud.loc['Sodio (ppm)', 'lim_critico_sup'] = 100
# Limite Warning  Oxidacion JOAP
limites_salud.loc['Oxidación JOAP (Abs/cm)', 'lim_warn_sup'] = 19
# Limite Critico  Oxidacion JOAP
limites_salud.loc['Oxidación JOAP (Abs/cm)', 'lim_critico_sup'] = 21
# Limite Warning Cobre
limites_desgaste.loc['Cobre (ppm)', 'lim_warn_sup'] = 6
# Limite Critico Cobre
limites_desgaste.loc['Cobre (ppm)', 'lim_critico_sup'] = 8
#Limite Warning Plomo
limites_desgaste.loc['Plomo (ppm)', 'lim_warn_sup'] = 4
#Limite Critico Plomo
limites_desgaste.loc['Plomo (ppm)', 'lim_critico_sup'] = 6

In [15]:
data_cleaned['SACODE_Salud']         = calificar_categoria(Salud,         limites_salud)
data_cleaned['SACODE_Desgaste']      = calificar_categoria(Desgaste,      limites_desgaste)
data_cleaned['SACODE_Contaminacion'] = calificar_categoria(Contaminacion, limites_contaminacion)

In [16]:
data_cleaned['SACODE_General'] = data_cleaned[['SACODE_Salud', 'SACODE_Desgaste', 'SACODE_Contaminacion']].apply(
    lambda x: 'Crítico' if 'Crítico' in x.values else ('Advertencia' if 'Advertencia' in x.values else 'Normal'),
    axis=1
)

### Estrategia de Balanceo

Para mitigar el desbalance se aplica la técnica SMOTE, que genera muestras sintéticas
para las clases minoritarias interpolando entre observaciones reales del espacio
de características, en lugar de simplemente duplicar registros existentes.

In [10]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split


# Variables predictoras y variable objetivo
X = data_cleaned.drop(columns=['Assigned Condition Rating',
                                'ACR_Homologado',
                                'Fault Effect',
                                'SACODE_Salud',
                                'SACODE_Desgaste',
                                'SACODE_Contaminacion',
                                'SACODE_General','Flota','Rule Based Rating'])

y = data_cleaned['Assigned Condition Rating']


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y       
)

print("=== Distribución antes de SMOTE (Train) ===")
print(y_train.value_counts())

# Aplicación de SMOTE exclusivamente sobre el conjunto de entrenamiento
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("\n=== Distribución después de SMOTE (Train) ===")
print(pd.Series(y_train_bal).value_counts())
print("\n=== Conjunto de prueba (sin modificar) ===")
print(y_test.value_counts())

=== Distribución antes de SMOTE (Train) ===
Assigned Condition Rating
Normal      2885
Warning      418
Critical     123
Name: count, dtype: int64

=== Distribución después de SMOTE (Train) ===
Assigned Condition Rating
Normal      2885
Warning     2885
Critical    2885
Name: count, dtype: int64

=== Conjunto de prueba (sin modificar) ===
Assigned Condition Rating
Normal      722
Warning     104
Critical     31
Name: count, dtype: int64


c:\Users\Daniel Martinez\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Daniel Martinez\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Daniel Martinez\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Daniel Martinez\AppData\Local\Pro

In [20]:
Salud_train = X_train_bal[['Boro (ppm)', 'Calcio (ppm)', 'Cinc (ppm)', 'Fosforado (ppm)',
                       'Magnesio (ppm)', 'Molibdeno (ppm)', 'Nitración JOAP (Abs/cm)',
                       'Oxidación JOAP (Abs/cm)', 'Sulfatación JOAP (Abs/cm)', 'Sodio (ppm)']]

Desgaste_train = X_train_bal[['Aluminio (ppm)', 'Cobre (ppm)', 'Cromo (ppm)', 'Estaño (ppm)',
                          'Hierro (ppm)', 'Plomo (ppm)',
                          'Oxidación JOAP (Abs/cm)']]

Contaminacion_train = X_train_bal[['Agua (%)', 'Dilución por combustible (%)',
                               'Hollín JOAP (Abs/cm)', 'Silicio (ppm)',
                               'Sodio (ppm)', 'Aluminio (ppm)']]


In [21]:
X_train_bal['SACODE_Salud']         = calificar_categoria(Salud_train,         limites_salud)
X_train_bal['SACODE_Desgaste']      = calificar_categoria(Desgaste_train,      limites_desgaste)
X_train_bal['SACODE_Contaminacion'] = calificar_categoria(Contaminacion_train, limites_contaminacion)

In [23]:
Salud_test = X_test[['Boro (ppm)', 'Calcio (ppm)', 'Cinc (ppm)', 'Fosforado (ppm)',
                       'Magnesio (ppm)', 'Molibdeno (ppm)', 'Nitración JOAP (Abs/cm)',
                       'Oxidación JOAP (Abs/cm)', 'Sulfatación JOAP (Abs/cm)', 'Sodio (ppm)']]

Desgaste_test = X_test[['Aluminio (ppm)', 'Cobre (ppm)', 'Cromo (ppm)', 'Estaño (ppm)',
                          'Hierro (ppm)', 'Plomo (ppm)',
                          'Oxidación JOAP (Abs/cm)']]

Contaminacion_test = X_test[['Agua (%)', 'Dilución por combustible (%)',
                               'Hollín JOAP (Abs/cm)', 'Silicio (ppm)',
                               'Sodio (ppm)', 'Aluminio (ppm)']]

In [24]:
X_test['SACODE_Salud']         = calificar_categoria(Salud_test,         limites_salud)
X_test['SACODE_Desgaste']      = calificar_categoria(Desgaste_test,      limites_desgaste)
X_test['SACODE_Contaminacion'] = calificar_categoria(Contaminacion_test, limites_contaminacion)

In [22]:
X_train_bal

,Unnamed: 0,Boro (ppm),Calcio (ppm),Cinc (ppm),Fosforado (ppm),Magnesio (ppm),Molibdeno (ppm),Agua (%),Dilución por combustible (%),Hollín JOAP (Abs/cm),...,Viscosidad @ 100°C (cSt),Aluminio (ppm),Cobre (ppm),Cromo (ppm),Estaño (ppm),Hierro (ppm),Plomo (ppm),SACODE_Salud,SACODE_Desgaste,SACODE_Contaminacion
0,3667,0.200000,1341.000000,1239.302808,1162.037844,125.019635,60.100000,0.0,0.000000,0.000000,...,14.960000,1.000000,1.000000,0.300000,0.000000,3.000000,2.200000,Normal,Normal,Normal
1,2691,47.520000,3530.000000,1419.662325,1152.073179,21.250000,34.230000,0.0,0.000000,2.000000,...,14.630000,2.370000,1.000000,0.510000,0.000000,6.000000,1.000000,Advertencia,Advertencia,Advertencia
2,2472,48.050000,3443.000000,1328.234553,1135.981777,20.980000,34.360000,0.0,0.000000,0.000000,...,15.160000,1.260000,0.190000,0.660000,0.044622,3.040000,0.000000,Advertencia,Normal,Normal
3,2838,45.980000,3444.966354,1246.839800,1038.889783,21.030000,35.020000,0.0,0.000000,0.000000,...,14.660000,1.000000,1.000000,0.090000,0.240000,3.000000,1.000000,Advertencia,Normal,Normal
4,3978,0.100000,1179.000000,1232.105806,1162.340280,123.585327,49.100000,0.0,0.000000,1.000000,...,14.950000,1.000000,1.000000,0.300000,0.000000,3.000000,1.100000,Normal,Normal,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8650,1876,38.214101,1828.747241,1428.106841,1225.723046,286.485725,63.933353,0.0,0.076639,22.968316,...,15.375475,0.972631,3.296721,0.452734,0.000000,11.660776,7.966251,Normal,Crítico,Advertencia
8651,2777,46.063606,3466.994788,1298.710451,1059.843691,25.396557,35.769331,0.0,0.000000,3.000000,...,15.621256,1.825259,1.000000,1.354073,0.000000,10.808320,1.000000,Crítico,Advertencia,Normal
8652,2270,41.546815,3383.957678,1129.628176,1045.484504,64.727263,33.270121,0.0,0.000000,8.248069,...,15.057085,1.583504,1.710203,0.648760,0.019323,5.152558,9.996660,Crítico,Crítico,Normal
8653,227,64.999477,1808.709779,1400.045463,1383.639820,375.952647,61.816959,0.0,0.068462,22.311189,...,14.861825,0.995595,0.448671,0.284895,0.000000,9.805072,0.413567,Advertencia,Normal,Advertencia
